# Motivace a způsob učení – explorační analýza

Autor: Ondřej Zetek

Explorace proměnných z oblasti *Motivace a způsob učení* (úvodní zpráva, slide 5)
před formulací analytické otázky. Notebook pouze popisuje data: dostupnost,
rozdělení, kódy chybějících hodnot a základní vztah k Science.
Pravidla zde zatím nehledáme.

Navazuje na nastavení z `03_vojtech_rodinne_zazemi.ipynb`: stejný Parquet,
stejné ošetření chybějících kódů, země GBR a vyšší Science = PV1SCIE nad 75. percentilem.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

DATA = PROJECT / "data/processed/student_full.parquet"
METADATA = PROJECT / "data/processed/student_full_metadata.json"
if not DATA.is_file() or not METADATA.is_file():
    raise FileNotFoundError("Najprv priprav plný Parquet podľa kroku 5 v README.")

metadata = json.loads(METADATA.read_text(encoding="utf-8"))
parquet = pq.ParquetFile(DATA)
if json.loads(parquet.schema_arrow.metadata[b"pisa_metadata"]) != metadata:
    raise ValueError("Parquet a metadata JSON nepatria k sebe. Použi oba súbory z rovnakého prevodu.")

by_name = {column["name"]: column for column in metadata["columns"]}
print(f"Záznamy: {parquet.metadata.num_rows:,}")
print(f"Premenné: {parquet.metadata.num_columns}")

Záznamy: 755,721
Premenné: 885


## Kandidátní proměnné

Výběr z codebooku (`PISA2025_Codebook.xlsx`, list STU). Odvozené indexy (WLE)
jsou spojité škály s průměrem OECD ≈ 0; ostatní jsou přímé odpovědi
nebo odvozené kategorie.

In [2]:
CANDIDATES = {
    # Postoje k přírodním vědám
    "JOYSCIE": "Radost z přírodních věd (index)",
    "EFFSCIE": "Sebedůvěra v přírodních vědách (index)",
    "STBELSCIE": "Názory na povahu vědy (index)",
    "ST094Q05NA": "Zajímá mě učit se o přírodních vědách",
    # Obecné motivační a sociálně-emoční rysy
    "CURIO": "Zvídavost (index)",
    "PERSEV": "Vytrvalost (index)",
    "ENPROBS": "Radost z řešení problémů (index)",
    "COGABIL": "Kognitivní přizpůsobivost (index)",
    "GOALSET": "Stanovování cílů (index)",
    "SELFREG": "Seberegulace (index)",
    "ST307Q10JA": "Snadno se vzdávám",
    # Způsob a objem učení
    "ENGSCIPR": "Zapojení do vědeckých postupů (index)",
    "ST432Q02DA": "Plánuji si, jak se budu učit",
    "ST071Q01NA": "Hodiny učení přírodních věd navíc týdně",
    "ST296Q03JA": "Čas na domácí úkoly z přírodních věd",
    "ST296Q04JA": "Čas na domácí úkoly celkem",
    "ST502Q01DA": "Individuální doučování přírodních věd",
    # Školní návyky a aspirace
    "SKIPPING": "Záškoláctví",
    "TARDYSD": "Pozdní příchody",
    "REPEAT": "Opakování ročníku",
    "EXPECEDU": "Očekávané nejvyšší vzdělání",
    "SISCO": "Jasná představa o budoucím povolání",
    "BSMJ": "Očekávaný profesní status (ISEI)",
    "EFFORT1": "Úsilí vložené do testu PISA (1–10)",
}

SCIENCE = [f"PV{i}SCIE" for i in range(1, 11)]
COLUMNS = ["CNT", "CNTSTUID", "W_FSTUWT"] + list(CANDIDATES) + SCIENCE

pd.DataFrame({
    "popis": CANDIDATES,
    "label_codebook": {name: by_name[name]["label"] for name in CANDIDATES},
    "chybejici_kody": {name: by_name[name]["missing_ranges"] for name in CANDIDATES},
})

,popis,label_codebook,chybejici_kody
JOYSCIE,Radost z přírodních věd (index),Students' enjoyment of science (WLE),"[{'lo': 95.0, 'hi': 99.0}]"
EFFSCIE,Sebedůvěra v přírodních vědách (index),Science self-efficacy (WLE),"[{'lo': 95.0, 'hi': 99.0}]"
STBELSCIE,Názory na povahu vědy (index),Students' beliefs about the nature of science ...,"[{'lo': 95.0, 'hi': 99.0}]"
ST094Q05NA,Zajímá mě učit se o přírodních vědách,How much do you disagree or agree with the sta...,"[{'lo': 5.0, 'hi': 9.0}]"
CURIO,Zvídavost (index),Students' curiousity (WLE),"[{'lo': 95.0, 'hi': 99.0}]"
PERSEV,Vytrvalost (index),Students' perseverance (WLE),"[{'lo': 95.0, 'hi': 99.0}]"
ENPROBS,Radost z řešení problémů (index),Enjoyment of problem-solving (WLE),"[{'lo': 95.0, 'hi': 99.0}]"
COGABIL,Kognitivní přizpůsobivost (index),Cognitive adaptability (WLE),"[{'lo': 95.0, 'hi': 99.0}]"
GOALSET,Stanovování cílů (index),Goal setting (WLE),"[{'lo': 95.0, 'hi': 99.0}]"
SELFREG,Seberegulace (index),Self-regulation (WLE),"[{'lo': 95.0, 'hi': 99.0}]"


## Chybějící hodnoty

Kódy chybějících hodnot (95–99, resp. 5–9) převádíme na `NaN` podle metadat,
stejně jako ve Vojtěchově notebooku. Srovnání GBR s celým souborem a s CZE ukazuje,
které proměnné jsou v GBR použitelné.

In [3]:
df = pd.read_parquet(DATA, columns=COLUMNS)

for name in COLUMNS:
    for interval in by_name[name]["missing_ranges"]:
        mask = df[name].between(interval["lo"], interval["hi"])
        df.loc[mask, name] = np.nan

missing = pd.DataFrame({
    "vse": df[list(CANDIDATES)].isna().mean(),
    "GBR": df.loc[df["CNT"].eq("GBR"), list(CANDIDATES)].isna().mean(),
    "CZE": df.loc[df["CNT"].eq("CZE"), list(CANDIDATES)].isna().mean(),
}).mul(100).round(1)

missing.sort_values("GBR")

,vse,GBR,CZE
EFFORT1,8.5,4.9,1.8
REPEAT,6.1,5.1,2.6
SKIPPING,4.9,5.6,3.4
TARDYSD,6.8,6.1,3.5
JOYSCIE,8.8,9.2,5.3
ST094Q05NA,9.1,9.3,5.4
EFFSCIE,10.3,10.0,5.8
ST296Q04JA,17.1,13.6,12.6
PERSEV,28.2,14.9,11.9
CURIO,19.3,15.3,11.9


## Rozsah: GBR

Science shrnujeme dvakrát: průměr deseti plausible values (pro popisné srovnání)
a binární `science_vyssi` podle PV1SCIE nad 75. percentilem GBR (stejná logika
jako u Vojtěcha, hranice je ale spočtena na celém GBR, ne na jeho kompletním vzorku).

In [4]:
gbr = df.loc[df["CNT"].eq("GBR")].copy()
gbr = gbr.loc[gbr[SCIENCE].notna().all(axis=1)]

gbr["science_prumer"] = gbr[SCIENCE].mean(axis=1)
threshold = gbr["PV1SCIE"].quantile(0.75)
gbr["science_vyssi"] = gbr["PV1SCIE"].gt(threshold)

print("Studenti GBR:", len(gbr))
print(f"Hranice vyšší Science (PV1SCIE, Q75): {threshold:.2f}")
print(f"Podíl vyšší Science: {gbr['science_vyssi'].mean():.1%}")

Studenti GBR: 15546
Hranice vyšší Science (PV1SCIE, Q75): 572.08
Podíl vyšší Science: 25.0%


## Rozdělení indexů

Některé indexy WLE mají výrazné *hromadné hodnoty* (velká část studentů
odpověděla na všechny položky stejně). Kategorizace podle kvartilů pak
nedává tři přibližně stejné skupiny – sloupec `nejcastejsi_podil` to ukazuje.

In [5]:
INDICES = [name for name in CANDIDATES if by_name[name]["label"].endswith("(WLE)")]

summary = gbr[INDICES].describe(percentiles=[0.25, 0.5, 0.75]).T
summary["nejcastejsi_hodnota"] = [gbr[n].round(2).mode().iloc[0] for n in INDICES]
summary["nejcastejsi_podil"] = [
    gbr[n].round(2).value_counts(normalize=True).iloc[0] for n in INDICES
]
summary.round(2)

,count,mean,std,min,25%,50%,75%,max,nejcastejsi_hodnota,nejcastejsi_podil
JOYSCIE,14121.0,0.05,1.07,-2.05,-0.63,0.49,0.49,2.06,0.49,0.31
EFFSCIE,13996.0,0.17,1.00,-2.75,-0.40,0.14,0.79,2.25,0.14,0.07
STBELSCIE,2596.0,0.05,0.90,-2.67,-0.29,0.03,0.60,1.80,-0.29,0.23
CURIO,13174.0,0.04,0.90,-2.68,-0.59,0.07,0.57,1.99,0.12,0.17
PERSEV,13224.0,-0.03,0.91,-2.65,-0.67,-0.07,0.38,2.28,0.38,0.17
ENPROBS,13070.0,0.02,0.97,-2.43,-0.64,-0.09,0.53,2.47,0.53,0.18
COGABIL,13000.0,-0.03,0.99,-3.05,-0.59,-0.05,0.45,2.16,-0.59,0.10
GOALSET,12805.0,0.09,1.00,-2.18,-0.37,0.19,0.81,2.43,-0.37,0.14
SELFREG,12951.0,-0.05,0.96,-2.83,-0.71,-0.06,0.55,2.21,-0.06,0.30
ENGSCIPR,12567.0,0.03,0.87,-1.73,-0.53,0.05,0.56,2.68,0.05,0.08


## Vztah k Science – korelace

Spearmanova korelace (přes pořadí) s průměrem PV. Jde jen o orientační
monotónní vztah; nelineární vzory (viz další tabulky) korelace nezachytí.

In [6]:
ranked = gbr[list(CANDIDATES) + ["science_prumer"]].rank()
correlations = pd.DataFrame({
    "spearman_rho": ranked[list(CANDIDATES)].corrwith(ranked["science_prumer"]),
    "n_platnych": gbr[list(CANDIDATES)].notna().sum(),
})
correlations["popis"] = pd.Series(CANDIDATES)
correlations.sort_values("spearman_rho").round(3)

,spearman_rho,n_platnych,popis
ST307Q10JA,-0.252,13146,Snadno se vzdávám
TARDYSD,-0.211,14592,Pozdní příchody
SKIPPING,-0.181,14671,Záškoláctví
REPEAT,-0.124,14757,Opakování ročníku
SISCO,-0.016,12618,Jasná představa o budoucím povolání
ST432Q02DA,0.033,12778,"Plánuji si, jak se budu učit"
ENGSCIPR,0.053,12567,Zapojení do vědeckých postupů (index)
ST071Q01NA,0.067,12310,Hodiny učení přírodních věd navíc týdně
ST296Q03JA,0.073,12930,Čas na domácí úkoly z přírodních věd
SELFREG,0.108,12951,Seberegulace (index)


## Vztah k Science – kategoriální proměnné

Pro každou kategorii: počet studentů, průměrná Science (průměr PV)
a podíl studentů s vyšší Science. Výchozí podíl je 25 %.

In [7]:
CATEGORICAL = [
    "ST094Q05NA", "ST307Q10JA", "ST432Q02DA", "ST296Q03JA", "ST296Q04JA",
    "ST502Q01DA", "SKIPPING", "TARDYSD", "REPEAT", "EXPECEDU", "SISCO", "EFFORT1",
]

def profile(column, groups=None):
    key = gbr[column] if groups is None else groups
    labels = {item["value"]: item["label"] for item in by_name[column]["value_labels"]}
    table = gbr.groupby(key, observed=True).agg(
        pocet=("science_prumer", "size"),
        science_prumer=("science_prumer", "mean"),
        vyssi_science_pct=("science_vyssi", "mean"),
    )
    table["vyssi_science_pct"] *= 100
    if groups is None:
        table.insert(0, "vyznam", [labels.get(value, "") for value in table.index])
    return table.round(1)

for name in CATEGORICAL:
    print(f"\n{name}: {CANDIDATES[name]}")
    display(profile(name))


ST094Q05NA: Zajímá mě učit se o přírodních vědách


,vyznam,pocet,science_prumer,vyssi_science_pct
ST094Q05NA,,,,
1.0,Strongly disagree,1834,462.6,13.8
2.0,Disagree,3459,479.6,16.3
3.0,Agree,6544,512.3,27.8
4.0,Strongly agree,2263,561.6,49.1



ST307Q10JA: Snadno se vzdávám


,vyznam,pocet,science_prumer,vyssi_science_pct
ST307Q10JA,,,,
1.0,Strongly disagree,2068,527.1,35.2
2.0,Disagree,4231,536.9,36.2
3.0,Neither agree or disagree,3521,498.4,23.3
4.0,Agree,2321,482.7,18.5
5.0,Strongly agree,1005,453.0,10.5



ST432Q02DA: Plánuji si, jak se budu učit


,vyznam,pocet,science_prumer,vyssi_science_pct
ST432Q02DA,,,,
1.0,Never or almost never,2157,510.4,30.5
2.0,Sometimes,5308,506.1,25.8
3.0,Often,3731,511.7,27.7
4.0,Very often,1582,519.0,31.3


... (14 outputs left out)


SISCO: Jasná představa o budoucím povolání


,vyznam,pocet,science_prumer,vyssi_science_pct
SISCO,,,,
0.0,not clear idea about future job,3075,511.2,27.1
1.0,clear idea about future job,9543,507.7,27.5



EFFORT1: Úsilí vložené do testu PISA (1–10)


,vyznam,pocet,science_prumer,vyssi_science_pct
EFFORT1,,,,
1.0,1,219,405.1,6.4
2.0,2,173,443.6,11.6
3.0,3,339,464.6,15.9
4.0,4,614,459.9,15.5
5.0,5,1254,467.0,15.3
6.0,6,2207,490.8,21.7
7.0,7,3562,512.4,28.2
8.0,8,3618,521.1,30.7
9.0,9,1689,531.6,35.2


## Vztah k Science – indexy po kvartilech

Indexy pro přehled dělíme na kvartily (`qcut`, duplicitní hranice se slučují).
Jde jen o exploraci, ne o finální kategorizaci pro procedury.

In [8]:
for name in INDICES:
    groups = pd.qcut(gbr[name], 4, duplicates="drop")
    print(f"\n{name}: {CANDIDATES[name]}")
    display(profile(name, groups))


JOYSCIE: Radost z přírodních věd (index)


,pocet,science_prumer,vyssi_science_pct
JOYSCIE,,,
"(-2.048, -0.633]",3804,466.3,13.9
"(-0.633, 0.49]",7467,505.8,25.2
"(0.49, 2.062]",2850,557.8,47.1



EFFSCIE: Sebedůvěra v přírodních vědách (index)


,pocet,science_prumer,vyssi_science_pct
EFFSCIE,,,
"(-2.747, -0.403]",3517,445.7,7.4
"(-0.403, 0.144]",4007,494.6,20.1
"(0.144, 0.791]",3045,534.9,34.7
"(0.791, 2.251]",3427,558.9,47.5



STBELSCIE: Názory na povahu vědy (index)


,pocet,science_prumer,vyssi_science_pct
STBELSCIE,,,
"(-2.67, -0.288]",1148,463.5,10.5
"(-0.288, 0.0263]",155,521.9,27.1
"(0.0263, 0.604]",667,525.9,30.6
"(0.604, 1.799]",626,548.7,39.8


... (10 outputs left out)


SELFREG: Seberegulace (index)


,pocet,science_prumer,vyssi_science_pct
SELFREG,,,
"(-2.83, -0.711]",3275,493.8,23.5
"(-0.711, -0.0604]",5828,504.7,24.9
"(-0.0604, 0.549]",993,540.1,39.7
"(0.549, 2.212]",2855,523.6,33.0



ENGSCIPR: Zapojení do vědeckých postupů (index)


,pocet,science_prumer,vyssi_science_pct
ENGSCIPR,,,
"(-1.7269999999999999, -0.528]",3143,494.0,20.3
"(-0.528, 0.0461]",3472,518.3,29.9
"(0.0461, 0.558]",2861,535.2,36.4
"(0.558, 2.677]",3091,505.5,28.3


## Hodiny učení navíc

`ST071Q01NA` je počet hodin (0–30+), ne škála – pro pravidla bude potřeba
vlastní seskupení. Zde jen rozdělení a průměrná Science.

In [9]:
hours = pd.cut(
    gbr["ST071Q01NA"],
    bins=[-0.5, 0.5, 3.5, 6.5, np.inf],
    labels=["0", "1-3", "4-6", "7+"],
)
profile("ST071Q01NA", hours)

,pocet,science_prumer,vyssi_science_pct
ST071Q01NA,,,
0,5053,506.6,25.5
1-3,4381,530.2,34.1
4-6,2006,507.2,27.6
7+,870,530.2,34.3


## Vzájemné korelace indexů

Silně korelované proměnné by se v pravidlech opakovaly jako téměř zaměnitelné
literály. Mezi indexy jsou korelace spíše mírné (do ~0,45).

In [10]:
gbr[INDICES].rank().corr().round(2)

,JOYSCIE,EFFSCIE,STBELSCIE,CURIO,PERSEV,ENPROBS,COGABIL,GOALSET,SELFREG,ENGSCIPR
JOYSCIE,1.00,0.35,0.29,0.31,0.27,0.33,0.15,0.24,0.19,0.20
EFFSCIE,0.35,1.00,0.29,0.29,0.26,0.34,0.28,0.24,0.18,0.26
STBELSCIE,0.29,0.29,1.00,0.30,0.20,0.18,0.17,0.19,0.17,0.11
CURIO,0.31,0.29,0.30,1.00,0.35,0.43,0.28,0.25,0.22,0.10
PERSEV,0.27,0.26,0.20,0.35,1.00,0.41,0.27,0.42,0.30,0.12
ENPROBS,0.33,0.34,0.18,0.43,0.41,1.00,0.29,0.30,0.21,0.19
COGABIL,0.15,0.28,0.17,0.28,0.27,0.29,1.00,0.19,0.29,0.12
GOALSET,0.24,0.24,0.19,0.25,0.42,0.30,0.19,1.00,0.22,0.20
SELFREG,0.19,0.18,0.17,0.22,0.30,0.21,0.29,0.22,1.00,0.12
ENGSCIPR,0.20,0.26,0.11,0.10,0.12,0.19,0.12,0.20,0.12,1.00


## Velikost kompletního vzorku

Kolik studentů GBR má vyplněné vybrané kombinace proměnných. Proměnné s vysokým
podílem chybějících hodnot (STBELSCIE, BSMJ, ST502Q01DA) výrazně zmenšují vzorek.

In [11]:
variants = {
    "postoje": ["JOYSCIE", "EFFSCIE", "ST094Q05NA"],
    "rysy": ["CURIO", "PERSEV", "GOALSET", "SELFREG", "ST307Q10JA"],
    "uceni": ["ST296Q04JA", "ST071Q01NA", "ST432Q02DA"],
    "navyky_aspirace": ["SKIPPING", "TARDYSD", "EXPECEDU"],
    "vse_bez_ridkych": [
        name for name in CANDIDATES if name not in ["STBELSCIE", "BSMJ", "ST502Q01DA"]
    ],
}

pd.DataFrame({
    name: {
        "pocet": int(gbr[columns].notna().all(axis=1).sum()),
        "podil_pct": round(100 * gbr[columns].notna().all(axis=1).mean(), 1),
    }
    for name, columns in variants.items()
}).T

,pocet,podil_pct
postoje,13811.0,88.8
rysy,12329.0,79.3
uceni,10786.0,69.4
navyky_aspirace,12484.0,80.3
vse_bez_ridkych,8383.0,53.9


## Poznámky k datům

Pozorování vyplývající z výstupů výše (popis dat, ne závěry):

- **STBELSCIE** chybí v GBR u ~83 % studentů (rotovaná položka) – pro GBR nepoužitelná.
  **BSMJ** (~39 %) a **ST502Q01DA** (~42 %) chybí také často.
- Nejsilnější monotónní vztah k Science mají postojové proměnné k vědám
  (EFFSCIE, JOYSCIE, ST094Q05NA) a aspirace (EXPECEDU, BSMJ); negativní
  návyky (ST307Q10JA, TARDYSD, SKIPPING, REPEAT).
- Několik vztahů je **nemonotónních**: domácí úkoly z přírodních věd (vrchol u 30 min–1 h denně),
  domácí úkoly celkem (vrchol u 1–3 h denně), hodiny učení navíc, ENGSCIPR a SELFREG
  (nejvyšší kvartil pod třetím), EFFORT1 (pokles u hodnoty 10), EXPECEDU (ISCED 8 pod ISCED 6–7).
  Korelační koeficient je u nich malý, ale rozdíly mezi kategoriemi nejsou.
- **Doučování** (ST502Q01DA) souvisí s *nižší* Science – pravděpodobně obrácená
  kauzalita (doučování dostávají slabší žáci).
- **JOYSCIE** má hromadnou hodnotu 0,49 (~31 % studentů) – kvartilové hranice
  nedají tři vyrovnané skupiny; lepší je kategorizovat podle odpovědí ST094 nebo
  podle hranic v hodnotách indexu.
- EFFORT1 je o testu samotném, ne o učení – spíš kontrolní proměnná než faktor.